In [6]:
using Pkg
Pkg.activate("../scripts/calval")
using IceFloeTracker
using Images
using Pkg
Pkg.activate("../scripts/calval")
using IceFloeTracker
using Images
using DataFrames
using CSVFiles
using CairoMakie
using StatsBase

  Activating project at `~/Documents/research/manuscripts/calval_tgrs/scripts/calval`
  Activating project at `~/Documents/research/manuscripts/calval_tgrs/scripts/calval`


The merge-floes method is based on the idea that, given two competing sets of labeled objects, we want to find the "best" segment in each and join them. We use two main metrics: circularity $C$ and mean boundary contrast $\bar \xi_b$. In addition, we examine two types of conflicts: members of relevant sets, and all other conflicts. Other conflicts are those where there is no shared centroid and mutual overlap is less than 50%. These other conflicts may include cases where no good segmentations exist, o 

In [8]:
# Load data
dataset = Watkins2026Dataset(;
    url="https://github.com/danielmwatkins/ice-floe-validation-dataset/",
    ref="v0.2",
    cache_dir="/tmp/Watkins2026",
    metadata_path="data/validation_dataset/validation_dataset.csv",
)

#### Run this to use the previously defined test/train split ####
prior_data = DataFrame(load("../data/train_test_split.csv"))
dataset.info[:, :training_sample] = parse.(Bool, prior_data[:, :training_sample])
dataset.info[:, :testing_sample] = parse.(Bool, prior_data[:, :testing_sample])
train_cases = filter(c -> c.training_sample, dataset) 
test_cases = filter(c -> c.testing_sample, dataset);

train_cases.info[:, :lookup_index] = 1:nrow(train_cases.info)
test_cases.info[:, :lookup_index] = 1:nrow(test_cases.info);

cluster_algo = IceDetectionBrightnessPeaksMODIS721(;
        band_7_max=0.16,
        possible_ice_threshold=0.6,
        join_method="union",
        minimum_prominence=0.01,
    )

cleanup_binary_params = (erosion_strel=strel_box((3, 3)), init_max_fill=100, conditional_max_fill=500
)
floe_splitting_params = (max_hole_fill=2000, max_distance=5, max_expand=3)

function clean_split_label(binary_mask, ice_mask, cloud_mask)
    return FSPipeline.dist_morph_split(
        FSPipeline.clean_binary_floes(binary_mask, ice_mask, cloud_mask; cleanup_binary_params...);
        floe_splitting_params...,
    )
end

# Workflows produce labeled arrays, not segmented images
function kmeans_workflow(preproc_gray, fc_masked, water_mask, ice_mask, cloud_mask; k=4, cluster_selection_algorithm=cluster_algo)
    masked_gray = apply_landmask(preproc_gray, water_mask)
    kmeans_result = kmeans_binarization(
        masked_gray, fc_masked; k=k, cluster_selection_algorithm=cluster_algo) .> 0
    return clean_split_label(kmeans_result, ice_mask, cloud_mask)
end

function adaptive_workflow(preproc_gray, land_mask, water_mask, ice_mask, cloud_mask; window_size=400, percentage=0)
    adaptive_result = binarize(preproc_gray, AdaptiveThreshold(; window_size=window_size, percentage=percentage)) .> 0
    apply_landmask!(adaptive_result, land_mask)
    apply_landmask!(adaptive_result, water_mask)
    return clean_split_label(adaptive_result .> 0, ice_mask, cloud_mask)    
end

adaptive_workflow (generic function with 1 method)

In [9]:
# Filter functions
function keep_labels!(img_indexmap, labels_list)    
    indices = component_indices(img_indexmap)
    labels = filter(r -> r > 0, unique(img_indexmap))
    for L in labels
        if L ∉ labels_list
            img_indexmap[indices[L]] .= 0
        end
    end
end

function filter_floes!(
    img_indexmap,
    coastal_buffer_mask,
    cloud_mask,
    falsecolor_image;
    min_floe_size=100,
    max_floe_size=90_000,
    min_circularity=0.4,
    expand_radius=15,
    filter_function=LogisticFilterFunction, # needs to operate on a dataframe
    prob_threshold=0.3
)
    # 1. Remove objects which overlap the coastal mask
    overlap = unique(img_indexmap[coastal_buffer_mask])
    indices = component_indices(img_indexmap)
    for L in overlap
        img_indexmap[indices[L]] .= 0
    end

    # 2. Remove objects outside the specified size bounds
    remove_small_segments!(img_indexmap, min_floe_size)
    remove_large_segments!(img_indexmap, max_floe_size)

    # Return blank image if no floes remain
    maximum(img_indexmap) == 0 && return img_indexmap

    # 3. Remove objects with low probability scores using filter_function
    # TODO: Generalize to allow other function types, inputs

    results_df = regionprops_table(img_indexmap;
        properties=[:label, :area, :convex_area, :perimeter, :bbox, :centroid,
                    :major_axis_length, :minor_axis_length, :orientation]
    )

    results_df[:, :length_scale] = results_df[:, :area] .^ 0.5
    
    # TODO: add component_circularity to the regionprops options
    results_df[:, :circularity] = 4 * π * results_df[:, :area] ./ results_df[:, :perimeter] .^ 2

    labels = results_df[:, :label]
    cloud_fractions = Dict(L => mean(cloud_mask[indices[L]]) for L in labels)
    results_df[:, :cloud_fraction] = [cloud_fractions[L] for L in labels]
    results_df[:, :cloudy] = results_df[:, :cloud_fraction] .> 0.5
    b2 = green.(falsecolor_image)
    b2_means = Dict(L => mean(b2[indices[L]]) for L in labels)
    
    bdry_indexmap = expand_labels(img_indexmap, expand_radius) .- img_indexmap
    bdry_indices = component_indices(bdry_indexmap)
    bdry_labels = intersect(labels, unique(bdry_indexmap))
    b2_bdry_means = Dict(L => mean(b2[bdry_indices[L]]) for L in bdry_labels)
    for L ∈ labels
        if L ∉ bdry_labels
            push!(b2_bdry_means, L => 0)
        end
    end
    results_df[:, :b2_reflectance_mean] = [b2_means[L] for L in labels]
    results_df[:, :b2_reflectance_bdry_mean] = [b2_bdry_means[L] for L in labels]
    results_df[:, :b2_bdry_contrast] = results_df[:, :b2_reflectance_mean] .- results_df[:, :b2_reflectance_bdry_mean]
    results_df[:, :prob] .= filter_function(results_df)
    labels = subset(results_df, :prob => r -> r .> prob_threshold)[:, :label]
    keep_labels!(img_indexmap, labels)

    # enforce minimum circularity
    labels = subset(results_df, :circularity => r -> r .> min_circularity)[:, :label]
    keep_labels!(img_indexmap, labels)
end

"""
    LogisticRegressionFilter(df; coefs)

Compute the probability for each DataFrameRow using the logistic function
with coefficients defined in `coefs`. Names should include "intercept" and
names of columns in `df`.

"""
function LogisticRegressionFilter(df;
    coefs=Dict(
        "intercept"           => -7.97726,
        "circularity"         => 3.28796,
        "cloudy"              => -2.21906,
        "length_scale"        => 0.0280868,
        "b2_bdry_contrast"    => 5.75358,
        "b2_reflectance_mean" => 6.75488,
        )
    )
    colnames = [x for x in keys(coefs)]
    b = [x for x in values(coefs)]
    df[:, :intercept] .= 1;
    return 1 ./ (1 .+ exp.(-Matrix(df[:, colnames]) * b))
end


LogisticRegressionFilter

In [10]:
case = train_cases

function fill_missing!(cases; template=Gray.(zeros(Bool, (400, 400))))
    for (idx, img) in enumerate(cases)
        if isnothing(img)
            cases[idx] = template
        end
    end
end

cloud_mask_function = Watkins2026CloudMask(
    band_7_threshold=0.16,
    band_2_threshold=0.34,
    opening_strel=strel_disk(3),
    dilation_strel=strel_disk(2)
)

ice_mask_function = IceDetectionBrightnessMidpoint(; minimum_reflectance=0.45)

@time begin
    tc_imgs = modis_truecolor.(case)
    fc_imgs = modis_falsecolor.(case)
    land_masks = modis_landmask.(case) .|> r -> r .> 0
    cloud_masks = cloud_mask_function.(fc_imgs)
    coastal_buffers = create_coastal_buffer_mask.(land_masks, [strel_box((25,25))])
    
    # joint_mask_buffers = ((x, y) -> x .|| y).(coastal_buffers, cloud_masks)
    
    # Mask images before k-means, preliminary ice mask application
    tc_masked = apply_landmask.(tc_imgs, land_masks)
    fc_masked = apply_landmask.(fc_imgs, land_masks)
    fc_buffered = apply_landmask.(fc_imgs, coastal_buffers)
    band_1 = fc_masked .|> r -> Gray.(blue.(r))
    ice_masks = ice_mask_function.(band_1) .|> r -> r .> 0
    # Masks have to be a full partition for this to work. So if the coastal buffer is used to make
    # the ice mask, it also must be used for the land mask here.
    water_masks = ((x, y, z) -> .!x .&& .!y .&& .!z).(land_masks, cloud_masks, ice_masks)
    
    # reference images
    binary_floes = validated_binary_floes.(case)
    fill_missing!(binary_floes)
end
nothing

 13.722101 seconds (46.21 M allocations: 4.032 GiB, 15.30% gc time, 55.74% compilation time: 4% of which was recompilation)


In [11]:
preproc_function = FSPipeline.Preprocess(
    diffusion_algorithm = PeronaMalikDiffusion(; λ=0.1, K=0.1, niters=5, g="exponential"),
    adapthisteq_params = (nbins=256, rblocks=2, cblocks=2, clip=5),
    unsharp_mask_params = (radius=50, amount=0.3, threshold=0.01),
)
tiles = get_tiles(tc_imgs[1], (400, 400)) # single tile
preproc_gray = preproc_function.(tc_imgs, land_masks, [tiles]);

In [12]:
adapt_labels = adaptive_workflow.(preproc_gray, land_masks, water_masks, ice_masks, cloud_masks; window_size=200, percentage=0);
kmeans_labels = kmeans_workflow.(preproc_gray, fc_buffered, water_masks, ice_masks, cloud_masks);

# Next Steps
1. Run the filter function, as it's set up now
2. Visualize the merging process (based on previous work)
3. Test whether I need to include more floes -- feature transform, e.g.
4. Test whether the right metrics are being used. How to test which are correct?